In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import xarray as xr
import numpy as np
import glob
import matplotlib.pyplot as plt

In [2]:
file_list = glob.glob("dataset/*.nc")
print("Files found:", len(file_list))

Files found: 10


In [3]:
class MethaneDataset(Dataset):
    def __init__(self, file_list):
        self.files = file_list

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        ds = xr.open_dataset(self.files[idx], group="PRODUCT")

        ch4 = ds['methane_mixing_ratio_bias_corrected'].values
        qa = ds['qa_value'].values

        # Cleaning
        ch4 = np.where(qa > 0.5, ch4, 0)

        # First time slice
        ch4 = ch4[0]

        # Normalize
        ch4 = (ch4 - np.min(ch4)) / (np.max(ch4) - np.min(ch4) + 1e-6)

        # Label (pseudo)
        mask = (ch4 > 0.6).astype(np.float32)

        # Shape (1, H, W)
        ch4 = np.expand_dims(ch4, axis=0)
        mask = np.expand_dims(mask, axis=0)

        return torch.tensor(ch4, dtype=torch.float32), torch.tensor(mask, dtype=torch.float32)

In [4]:
class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        def block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.ReLU(),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.ReLU()
            )

        self.enc1 = block(1, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = block(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = block(128, 256)

        self.up1 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec1 = block(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = block(128, 64)

        self.out = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))

        d1 = self.up1(e3)
        d1 = torch.cat([d1, e2], dim=1)
        d1 = self.dec1(d1)

        d2 = self.up2(d1)
        d2 = torch.cat([d2, e1], dim=1)
        d2 = self.dec2(d2)

        return torch.sigmoid(self.out(d2))

In [5]:
dataset = MethaneDataset(file_list)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCELoss()

In [6]:
import torch
import torch.nn.functional as F
import xarray as xr
import numpy as np

class MethaneDataset:
    def __init__(self, file_list):
        self.files = file_list

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        ds = xr.open_dataset(self.files[idx], group="PRODUCT")

        # ✅ DEFINE VARIABLES FIRST
        ch4 = ds['methane_mixing_ratio_bias_corrected'].values
        qa = ds['qa_value'].values

        # Cleaning
        ch4 = np.where(qa > 0.5, ch4, 0)

        # Take first slice
        ch4 = ch4[0]

        # Normalize
        ch4 = (ch4 - np.min(ch4)) / (np.max(ch4) - np.min(ch4) + 1e-6)

        # Label
        mask = (ch4 > 0.6).astype(np.float32)

        # Convert to tensor
        ch4 = torch.tensor(ch4, dtype=torch.float32).unsqueeze(0)
        mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)

        # ✅ RESIZE (IMPORTANT)
        ch4 = F.interpolate(ch4.unsqueeze(0), size=(128,128), mode='bilinear')[0]
        mask = F.interpolate(mask.unsqueeze(0), size=(128,128), mode='nearest')[0]

        return ch4, mask

In [7]:
import torch
import torch.nn.functional as F
import xarray as xr
import numpy as np
from torch.utils.data import Dataset

class MethaneDataset(Dataset):
    def __init__(self, file_list):
        self.files = file_list

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        ds = xr.open_dataset(self.files[idx], group="PRODUCT")

        # 🔥 1. Load data
        ch4 = ds['methane_mixing_ratio_bias_corrected'].values
        qa = ds['qa_value'].values

        # 🔥 2. Clean
        ch4 = np.where(qa > 0.5, ch4, 0)

        # 🔥 3. Take first slice
        ch4 = ch4[0]

        # 🔥 4. Normalize
        ch4 = (ch4 - np.min(ch4)) / (np.max(ch4) - np.min(ch4) + 1e-6)

        # 🔥 5. Create label
        mask = (ch4 > 0.6).astype(np.float32)

        # 🔥 6. Convert to tensor
        ch4 = torch.tensor(ch4, dtype=torch.float32).unsqueeze(0)
        mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)

        # 🔥 7. FORCE RESIZE (MOST IMPORTANT)
        ch4 = F.interpolate(ch4.unsqueeze(0), size=(128, 128), mode='bilinear', align_corners=False)
        mask = F.interpolate(mask.unsqueeze(0), size=(128, 128), mode='nearest')

        ch4 = ch4.squeeze(0)
        mask = mask.squeeze(0)

        return ch4, mask

In [8]:
epochs = 5

for epoch in range(epochs):
    total_loss = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        pred = model(x)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

RuntimeError: Sizes of tensors must match except in dimension 1. Expected size 106 but got size 107 for tensor number 1 in the list.